# Q1: Used Cars Data Preparation

Preparing a used-cars dataset for downstream modeling. The target variable is `Price` (in lakhs). Steps below: handle missing values, strip units off numeric columns, one-hot encode the categoricals, add a car-age feature, then run a set of select / filter / rename / mutate / arrange / group_by style operations.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)

df = pd.read_csv('data/used_cars.csv').drop(columns=['Unnamed: 0'])
df.shape

(5847, 13)

In [2]:
df.head()

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


## (a) Missing values (4 pts)

Counts per column below. `New_Price` is missing in about 86% of rows, so I drop that column instead of imputing - imputing a value we only observe for 14% of cars would add more noise than signal. For `Engine`, `Power`, `Seats`, `Mileage` the missingness is under 1%, so I keep them and impute.

`Mileage`, `Engine`, and `Power` are stored as strings with unit suffixes, so I convert them to numbers first (step b) and then impute with the median. Median is more robust to the long right tail in these automotive specs than the mean. `Seats` is a small integer so I impute with the mode (5, the most common seat count for passenger cars).

In [3]:
missing = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct': (df.isna().mean() * 100).round(2),
})
missing[missing['n_missing'] > 0]

,n_missing,pct
Mileage,2,0.03
Engine,36,0.62
Power,36,0.62
Seats,38,0.65
New_Price,5032,86.06


In [4]:
df = df.drop(columns=['New_Price'])

## (b) Strip units from numeric columns (4 pts)

`Mileage` (kmpl or km/kg), `Engine` (CC), `Power` (bhp) come in as strings. I strip the unit suffix and convert to float. A handful of `Power` values are the literal string `null bhp`; those become NaN and get imputed with the others.

In [5]:
def strip_unit(s):
    return pd.to_numeric(s.astype(str).str.split().str[0], errors='coerce')

for col in ['Mileage', 'Engine', 'Power']:
    df[col] = strip_unit(df[col])

df[['Mileage', 'Engine', 'Power']].describe()

,Mileage,Engine,Power
count,5845.000000,5811.000000,5811.000000
mean,18.158496,1631.552573,113.803144
std,4.358246,601.972587,53.896719
min,0.000000,72.000000,34.200000
25%,15.260000,1198.000000,78.000000
50%,18.190000,1497.000000,98.600000
75%,21.100000,1991.000000,139.010000
max,28.400000,5998.000000,560.000000


In [6]:
for col in ['Mileage', 'Engine', 'Power']:
    df[col] = df[col].fillna(df[col].median())
df['Seats'] = df['Seats'].fillna(df['Seats'].mode().iloc[0])

df.isna().sum().sum()

0

## (c) One-hot encode Fuel_Type and Transmission (4 pts)

`pd.get_dummies` with integer dtype so the encoded columns are 0/1 rather than boolean. I keep all levels (no drop_first) so the encoding is readable in the output; a modeler can drop a reference level later if they need to avoid collinearity.

In [7]:
df = pd.get_dummies(df, columns=['Fuel_Type', 'Transmission'], dtype=int)
[c for c in df.columns if c.startswith(('Fuel_Type_', 'Transmission_'))]

['Fuel_Type_Diesel',
 'Fuel_Type_Electric',
 'Fuel_Type_Petrol',
 'Transmission_Automatic',
 'Transmission_Manual']

## (d) New feature: Car_Age (4 pts)

Year of sale is not recorded directly, but the dataset was released around 2019, so I compute `Car_Age = 2019 - Year`. A 2010 car is 9 years old at the time of listing. This is more useful to a model than the raw year.

In [8]:
df['Car_Age'] = 2019 - df['Year']
df[['Year', 'Car_Age']].describe()

,Year,Car_Age
count,5847.000000,5847.000000
mean,2013.448435,5.551565
std,3.194949,3.194949
min,1998.000000,0.000000
25%,2012.000000,3.000000
50%,2014.000000,5.000000
75%,2016.000000,7.000000
max,2019.000000,21.000000


## (e) select / filter / rename / mutate / arrange / group_by (4 pts)

The six dplyr verbs applied in order. Each step is a separate cell with a short note about what it does.

**select** - keep a focused set of columns.

In [9]:
sub = df[['Name', 'Location', 'Year', 'Car_Age',
          'Kilometers_Driven', 'Mileage', 'Engine', 'Power',
          'Seats', 'Owner_Type', 'Price']]
sub.head()

,Name,Location,Year,Car_Age,Kilometers_Driven,Mileage,Engine,Power,Seats,Owner_Type,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,4,41000,19.67,1582.0,126.20,5.0,First,12.50
1,Honda Jazz V,Chennai,2011,8,46000,13.00,1199.0,88.70,5.0,First,4.50
2,Maruti Ertiga VDI,Chennai,2012,7,87000,20.77,1248.0,88.76,7.0,First,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,6,40670,15.20,1968.0,140.80,5.0,Second,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,6,86999,23.08,1461.0,63.10,5.0,First,3.50


**filter** - cars 5 years old or newer and priced under 20 lakh.

In [10]:
f = sub[(sub['Car_Age'] <= 5) & (sub['Price'] < 20)]
f.shape

(2665, 11)

**rename** - clearer column names.

In [11]:
r = f.rename(columns={'Kilometers_Driven': 'Km_Driven',
                      'Owner_Type': 'Owners',
                      'Price': 'Price_Lakh'})
r.columns.tolist()

['Name',
 'Location',
 'Year',
 'Car_Age',
 'Km_Driven',
 'Mileage',
 'Engine',
 'Power',
 'Seats',
 'Owners',
 'Price_Lakh']

**mutate** - price per km, and power divided by seats.

In [12]:
r = r.assign(
    Price_Per_Km=r['Price_Lakh'] * 1e5 / r['Km_Driven'].clip(lower=1),
    Power_Per_Seat=r['Power'] / r['Seats'],
)
r[['Name', 'Price_Lakh', 'Km_Driven', 'Price_Per_Km', 'Power_Per_Seat']].head()

,Name,Price_Lakh,Km_Driven,Price_Per_Km,Power_Per_Seat
0,Hyundai Creta 1.6 CRDi SX Option,12.50,41000,30.487805,25.2400
5,Toyota Innova Crysta 2.8 GX AT 8S,17.50,36000,48.611111,21.4375
8,Maruti Ciaz Zeta,9.95,25692,38.728009,20.6500
10,Maruti Swift VDI BSIV,5.60,64424,8.692413,14.8000
13,Mitsubishi Pajero Sport 4X4,15.00,110000,13.636364,25.0800


**arrange** - sort by price, highest first.

In [13]:
r.sort_values('Price_Lakh', ascending=False).head(10)

,Name,Location,Year,Car_Age,Km_Driven,Mileage,Engine,Power,Seats,Owners,Price_Lakh,Price_Per_Km,Power_Per_Seat
3665,Toyota Innova Crysta 2.8 ZX AT,Kochi,2017,2,26023,11.36,2755.0,171.50,7.0,First,19.97,76.739807,24.500000
4333,Land Rover Freelander 2 SE,Kochi,2014,5,61730,12.39,2179.0,147.51,5.0,First,19.94,32.301960,29.502000
847,Toyota Innova Crysta 2.8 ZX AT,Kochi,2018,1,41736,11.36,2755.0,171.50,7.0,First,19.92,47.728580,24.500000
1379,Toyota Innova Crysta 2.4 VX MT,Kochi,2018,1,8899,13.68,2393.0,147.80,7.0,First,19.92,223.845376,21.114286
197,Audi Q3 2012-2015 35 TDI Quattro Premium Plus,Delhi,2015,4,64103,15.73,1968.0,174.33,5.0,First,19.90,31.043789,34.866000
4377,BMW 3 Series 320d Sport Line,Mumbai,2014,5,23000,18.88,1995.0,184.00,5.0,First,19.90,86.521739,36.800000
4640,Mercedes-Benz B Class B200 CDI,Bangalore,2016,3,36610,15.00,2143.0,107.30,5.0,First,19.90,54.356733,21.460000
706,BMW 3 Series 320d Luxury Line,Kochi,2015,4,58390,18.88,1995.0,184.00,5.0,First,19.86,34.012673,36.800000
2914,Toyota Fortuner 4x2 Manual,Mumbai,2015,4,31000,13.00,2982.0,168.50,7.0,First,19.85,64.032258,24.071429
5837,Toyota Camry Hybrid,Mumbai,2015,4,33500,19.16,2494.0,158.20,5.0,First,19.75,58.955224,31.640000


**summarize with group_by** - city-level averages.

In [14]:
(r.groupby('Location')
  .agg(n=('Price_Lakh', 'size'),
       avg_price=('Price_Lakh', 'mean'),
       median_km=('Km_Driven', 'median'),
       avg_mileage=('Mileage', 'mean'))
  .round(2)
  .sort_values('avg_price', ascending=False))

,n,avg_price,median_km,avg_mileage
Location,,,,
Coimbatore,361,8.23,38461.0,19.50
Bangalore,107,7.90,48808.0,18.90
Kochi,437,7.72,41380.0,19.72
Ahmedabad,105,7.53,52000.0,20.56
Mumbai,371,7.41,29000.0,18.59
Delhi,227,7.23,50000.0,19.90
Hyderabad,264,7.11,55511.5,21.23
Pune,212,7.04,48000.0,20.29
Chennai,155,6.99,46663.0,20.55


## Cleaned dataset

The cleaned frame is written to `data/used_cars_clean.csv` for downstream use.

In [15]:
df.to_csv('data/used_cars_clean.csv', index=False)
df.shape

(5847, 16)